# Search Results → AnnData

Converts raw Spectronaut output into [AnnData](https://anndata.readthedocs.io) objects for downstream analysis. This is the second step in the data-object creation pipeline (run after `01_metadata.ipynb`).

**Inputs**
- Spectronaut BGS Factory Report (Normal) TSV — main single-cell run
- Spectronaut Factory Report TSVs — quantitative QC mixture runs (one per site)
- `metadata.parquet` — sample-level annotations produced by `01_metadata.ipynb`

**Outputs** (written to `Data/AnnData/` and `Data/Parquet/`)
| File | Description |
|---|---|
| `adata_precursor.h5ad` | Precursor-level quantification, human samples |
| `adata_protein.h5ad` | Protein-level quantification, human samples |
| `adata_precursor_qc.h5ad` | Precursor-level quantification, QC mixtures |
| `adata_protein_qc.h5ad` | Protein-level quantification, QC mixtures |
| `search_results.parquet` | Filtered long-format search results, human samples |
| `search_results_qc.parquet` | Filtered long-format search results, QC mixtures |

---

## Imports

In [24]:
import os
from tqdm import tqdm
import pandas as pd
import numpy as np
from pathlib import Path
from glob import glob

# TODO: Fix ProteoBench dependency
from proteobench.io.parsing.parse_ion import _LOAD_FUNCTIONS

from SCI_py.data.parse import df_to_anndata, align_obs_across_anndata
from SCI_py.metadata.parse import FILENAME_CONVERTERS, get_institute

tqdm.pandas()

## 0. Path Configuration

### Output paths

Resolved relative to the notebook location. Override with the `SCI_DATA_DIR` environment variable when running outside the default directory structure.

In [25]:
NOTEBOOK_DIR = Path().resolve()
ROOT_OUT      = NOTEBOOK_DIR / "../../Data"
ROOT_OUT      = ROOT_OUT.resolve()
ROOT_OUT.mkdir(parents=True, exist_ok=True)

# Allow user override via environment variable
ROOT_OUT = Path(os.environ.get("SCI_DATA_DIR", ROOT_OUT))

ROOT_OUT.mkdir(parents=True, exist_ok=True)
print(f"Output directory: {ROOT_OUT}")

Output directory: /home/sam/SCI/SCI/Data


### Input paths

Points to the Spectronaut result files on the compute node (`conode53`) and the metadata parquet generated by the previous notebook. A warning is printed if the metadata file is missing, as the merge step will fail without it.

In [26]:
# For the search result output from Spectronaut
ROOT_IN = '/public/conode53/sam/SCI'
result_path = os.path.join(
    ROOT_IN,
    'full_quant_tables',
    'spectronaut',
    'merge_directdia',
    '20260307_124034_Bogdan_12082025',
    'Bogdan_12082025_Report_BGS Factory Report (Normal).tsv'
)

# For the quantitative QCs run separately
qc_root = '/public/conode53/sam/SCI_paper/QC_Quant'
qc_paths = glob(os.path.join(qc_root, '*', 'Spectronaut', '4', '*Factory Report (Normal).tsv'))

# For the metadata file generated in previous notebook
METADATA_PATH = os.path.join(
    ROOT_OUT,
    'metadata',
    'metadata.parquet'
)
if not os.path.exists(METADATA_PATH):
    print("Warning: Run 01_metadata.ipynb first or change the METADATA_PATH variable.")

## 1. Load Data

Parse Spectronaut reports and join them with sample metadata. The loading function standardises modification notation to UNIMOD format and constructs a canonical precursor ID (`FG.PrecursorIon`) used as the feature index at the precursor level.

### 1.1. Search Results

`load_results_spectronaut` wraps the ProteoBench parser and applies three normalisation steps:
1. **Institute / plate ID** — extracted from the raw filename via `get_institute`.
2. **Sample name** — converted to a canonical lowercase identifier using site-specific `FILENAME_CONVERTERS`.
3. **Modification notation** — Spectronaut labels (`[Oxidation (M)]`, etc.) are mapped to UNIMOD accessions for interoperability.

In [27]:
def load_results_spectronaut(path):
    report = _LOAD_FUNCTIONS['Spectronaut'](
        input_csv=path
    )

    # Parse sample name and sample_ids
    report['plate_id'] = report['R.FileName'].progress_apply(
        get_institute
    )
    report['sample_name'] = report.progress_apply(
        lambda x: FILENAME_CONVERTERS[x['plate_id']](x['R.FileName']+'.RAW'),
        axis=1
    )
    report['sample_name'] = report['sample_name'].progress_apply(lambda x: x.lower())

    # Parse the modification annotation
    report['FG.PrecursorIon'] = report.progress_apply(
        lambda x: (
            str(x['FG.LabeledSequence'])
                .strip('_')
                .replace('[Oxidation (M)]', '[UNIMOD:35]')
                .replace('[Acetyl (Protein N-term)]', '[UNIMOD:1]')
                .replace('[Carbamidomethyl (C)]', '[UNIMOD:4]') + '/' + str(x['FG.Charge'])
        ),
        axis=1
    )
    return report

#### 1.1.1. Human samples (+ blanks)

Main SCI dataset: a single merged Spectronaut report covering all participating institutes. Blanks are retained at this stage and filtered downstream based on the `CellInBlank` metadata flag.

In [ ]:
report = load_results_spectronaut(result_path)

100%|██████████| 21660835/21660835 [02:09<00:00, 167499.73it/s]


#### 1.1.2. Quantitative QC (mixtures)

Each site ran a set of mixture-based QC samples independently (one Spectronaut export per site, stored under `QC_Quant/`). These are parsed with the same loader and concatenated into a single dataframe for joint analysis.

In [ ]:
qcs = []
for p in tqdm(qc_paths):
    qc = load_results_spectronaut(p)
    qcs.append(qc)
report_qc = pd.concat(qcs)

100%|██████████| 9/9 [01:18<00:00,  8.73s/it]


### 1.2. Merge with Metadata

Join search results with sample-level metadata on `plate_id` + `sample_name`. This attaches morphological measurements (diameter, circularity, …), experimental annotations (gradient time, plate arrival), and quality flags (doublet, dying, …) to every quantified precursor row.

In [ ]:
metadata = pd.read_parquet(
    METADATA_PATH
)

df = report.merge(metadata, on=['plate_id', 'sample_name'])
df_qc = report_qc.merge(metadata, on=['plate_id', 'sample_name'])

print(df.shape)
print(df_qc.shape)

(2505299, 150)


### 1.3. Subdivide Columns

Partition the merged dataframe columns into logical groups before pivoting to AnnData. This makes the mapping to AnnData slots explicit:

| Group | AnnData slot | Purpose |
|---|---|---|
| `cols_protpept_metadata` + `cols_idconf` + `cols_peptfeatures` | `adata.var` | Feature-level annotations |
| `cols_metadata` | `adata.obs` | Sample-level annotations |
| `cols_quant` | `adata.X` (via pivot) | Quantitative intensities |

Only the columns used downstream are retained, reducing memory footprint before the pivot.

In [32]:
cols_protpept_metadata = [
    'PG.ProteinGroups',
    'PG.ProteinAccessions',
    'PEP.StrippedSequence',
    'PEP.IsProteotypic',
    'EG.ModifiedSequence',
    'FG.Charge',
    'FG.LabeledSequence',
    'FG.PrecursorIon'
]
cols_idconf = [
    'PG.Qvalue',
    'FG.Qvalue',
    'EG.GlobalPrecursorQvalue'
]
cols_quant = [
    'PG.Quantity',
    'EG.TargetQuantity (Settings)',
    'EG.ReferenceQuantity (Settings)',
    'EG.TotalQuantity (Settings)',
    'FG.Quantity',
]
cols_peptfeatures = [
    # Retention time
    'EG.ApexRT',
    'EG.RTPredicted',
    'EG.iRTEmpirical',
    'EG.iRTPredicted',
    
    # MS2 features
    'FG.PrecWindowNumber',
    'FG.CScore',
]
cols_metadata = [
    'sample_name',
    'plate_id',
    'plate_idx',
    'source name',
    'comment[X position]',
    'comment[Y position]',
    'comment[Plate Number]',
    'comment[Gradient time]',
    'comment[thawing timepoint]',
    'comment[plate arrival]',
    'institute_name',
    'Reinjection',
    'Diameter',
    'Elongation',
    'Circularity',
    'Doublet',
    'Dividing',
    'Dying',
    'CellInBlank',
    'comment[plate arrival]_date',
    'comment[plate arrival]_time', 'comment[plate arrival]_datetime',
    'comment[Gradient time]_date', 'comment[Gradient time]_time',
    'comment[Gradient time]_datetime', 'instrument_type'
]

col_sample_index = ['sample_id']

var_cols = cols_protpept_metadata + cols_idconf + cols_peptfeatures

In [ ]:
df = df.loc[
    :,
    (
        cols_protpept_metadata +
        col_sample_index +
        cols_idconf +
        cols_quant +
        cols_peptfeatures +
        cols_metadata
    )
]

df_qc = df_qc.loc[
    :,
    (
        cols_protpept_metadata +
        col_sample_index +
        cols_idconf +
        cols_quant +
        cols_peptfeatures +
        cols_metadata
    )
]

## 2. Create AnnData Objects

Pivot the long-format dataframes into AnnData's `obs × var` matrix layout. Precursor and protein objects are built separately because they use different feature indices (`FG.PrecursorIon` vs `PG.ProteinGroups`) and different quantification columns.

### 2.1. Parse Dataframe to AnnData

`df_to_anndata` pivots the long-format report into a sparse `obs × var` intensity matrix (`adata.X`). `align_obs_across_anndata` then ensures the precursor and protein objects share an identical observation index — required for any joint analysis or cross-level comparisons later.

In [20]:
adata_precursor, X_precursor = df_to_anndata(
    df,
    feature_col = 'FG.PrecursorIon',
    value_col   = 'EG.TotalQuantity (Settings)',
    sample_col  = 'sample_id',
    obs_cols    = cols_metadata,
    var_cols    = var_cols,
)

adata_protein, X_protein = df_to_anndata(
    df,
    feature_col = 'PG.ProteinGroups',
    value_col   = 'PG.Quantity',
    sample_col  = 'sample_id',
    obs_cols    = cols_metadata,
    var_cols    = var_cols,
)

adata_precursor, adata_protein = align_obs_across_anndata(
    adata_precursor,
    X_precursor,
    adata_protein,
    X_protein
)

In [34]:
adata_precursor_qc, X_precursor_qc = df_to_anndata(
    df_qc,
    feature_col = 'FG.PrecursorIon',
    value_col   = 'EG.TotalQuantity (Settings)',
    sample_col  = 'sample_id',
    obs_cols    = cols_metadata,
    var_cols    = var_cols,
)

adata_protein_qc, X_protein_qc = df_to_anndata(
    df_qc,
    feature_col = 'PG.ProteinGroups',
    value_col   = 'PG.Quantity',
    sample_col  = 'sample_id',
    obs_cols    = cols_metadata,
    var_cols    = var_cols,
)

adata_precursor_qc, adata_protein_qc = align_obs_across_anndata(
    adata_precursor_qc,
    X_precursor_qc,
    adata_protein_qc,
    X_protein_qc
)

### 2.2. Add Log-Normalised Layers

Store a `log2`-transformed copy of the raw intensities as a separate layer (`X_log2`). The raw intensities are kept in `adata.X` so both scales are available without recomputation. Log-transformation is applied before saving because most downstream statistical and visualisation steps operate on the log scale.

In [ ]:
adata_precursor.layers['X_log2'] = np.log2(adata_precursor.X)
adata_protein.layers['X_log2'] = np.log2(adata_protein.X)

adata_precursor_qc.layers['X_log2'] = np.log2(adata_precursor_qc.X)
adata_precursor_qc.layers['X_log2'] = np.log2(adata_precursor_qc.X)

## 3. Save

Write all four AnnData objects to `Data/AnnData/` (`.h5ad` format) and the filtered long-format dataframes to `Data/Parquet/`. The parquet files preserve the full column set for any subsequent ad-hoc queries without needing to re-parse the raw Spectronaut TSVs.

> `convert_strings_to_categoricals=False` is set explicitly to avoid silent dtype changes that can break downstream string operations.

In [ ]:

ANNDATA_OUT = os.path.join(
    ROOT_OUT,
    'AnnData'
)
PARQUET_OUT = os.path.join(
    ROOT_OUT,
    'Parquet'
)
Path(ANNDATA_OUT).mkdir(parents=True, exist_ok=True)
Path(PARQUET_OUT).mkdir(parents=True, exist_ok=True)

adata_precursor.write_h5ad(
    os.path.join(ANNDATA_OUT, 'adata_precursor.h5ad'),
    convert_strings_to_categoricals=False
)
adata_protein.write_h5ad(
    os.path.join(ANNDATA_OUT, 'adata_protein.h5ad'),
    convert_strings_to_categoricals=False
)
adata_precursor_qc.write_h5ad(
    os.path.join(ANNDATA_OUT, 'adata_precursor_qc.h5ad'),
    convert_strings_to_categoricals=False
)
adata_protein_qc.write_h5ad(
    os.path.join(ANNDATA_OUT, 'adata_protein_qc.h5ad'),
    convert_strings_to_categoricals=False
)

df.to_parquet(
    os.path.join(PARQUET_OUT, 'search_results.parquet')
)
df_qc.to_parquet(
    os.path.join(PARQUET_OUT, 'search_results_qc.parquet')
)